# Cycle 6 : Test undersampling


In [1]:
### Import des modules
%load_ext autoreload
%autoreload 2

import pandas as pd
import utils
from sklearn.ensemble import RandomForestClassifier

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [2]:
data = pd.read_csv('data/rafined/employees.csv', sep=',')

In [4]:
train_data = utils.split_train_data(data, 'a_quitte_l_entreprise')

In [5]:
from sklearn.preprocessing import RobustScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('robust_scaler', RobustScaler(), ['revenu_mensuel']),
        ('minmax_scaler', MinMaxScaler(), ['age']),
        ('encoder', OneHotEncoder(), ['statut_marital', 'departement', 'poste', 'domaine_etude']),
    ],
    remainder='passthrough'
)

In [11]:
from sklearn.model_selection import FixedThresholdClassifier
from technova_features import TechNovaFeatureEngineering
from imblearn.under_sampling import NearMiss
from imblearn.pipeline import Pipeline

random_forest_model = RandomForestClassifier(random_state=42)

threshold_model = FixedThresholdClassifier(
    estimator=random_forest_model,
    threshold=0.3,
    response_method="predict_proba"
)

pipeline = Pipeline([
    ('features', TechNovaFeatureEngineering()),
    ('preprocessor', preprocessor),
    ('sampling', NearMiss(version=1)),
    ('model', threshold_model),
])

utils.benchmark(pipeline, train_data)

CrossValidation Results : 
{'fit_time': array([0.08157086, 0.08512807, 0.08568501, 0.08087206, 0.07499194]), 'score_time': array([0.01380491, 0.01156592, 0.01421714, 0.01165104, 0.011024  ]), 'test_recall': array([0.95      , 0.94871795, 0.8974359 , 0.95      , 0.975     ]), 'test_f1': array([0.30039526, 0.30204082, 0.29535865, 0.31404959, 0.31325301])}
Recall moyen : 0.9442307692307692
F1-Score moyen : 0.3050194643715217
Training Résults : 
Recall moyen : 0.8974358974358975
F1-Score moyen : 0.22950819672131148
[[ 24 231]
 [  4  35]]


In [10]:
utils.train(pipeline, train_data, 0.5)

Training Résults : 
Recall moyen : 0.717948717948718
F1-Score moyen : 0.21374045801526717
[[ 60 195]
 [ 11  28]]
Train F1-Score: 0.3697478991596639
Test F1-Score: 0.21374045801526717
L'AUC du modèle est : 0.5135243841126194
